# AIoT HW2: 多元線性回歸與 CRISP-DM 流程實踐

> **課程**：物聯網數據分析與應用 (AIoT Data Analysis and Applications)  
> **主題**：Kaggle Dataset × Multiple Linear Regression × CRISP-DM 6 大階段  
> **核心重點**：無資料洩漏 (No Data Leakage)、Filter 特徵選擇、95% 預測區間 (Prediction Interval)  
> **作者**：kelp11211  
> **儲存庫**：https://github.com/kelp11211/AIoT_HW2

## Phase 1: Business Understanding (商業理解)
- **目標**：以 20 個環境與水文治理指標預估洪水機率 (`FloodProbability`)，在強降雨發生前提供即時防災調度。
- **成功標準**：RMSE 與 MAE 顯著超越 Baseline (DummyRegressor)，決定係數 $R^2 \ge 0.84$。

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
BASE_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from src.data_loader import load_dataset
from src.preprocessing import split_data, build_preprocessor
from src.feature_selection import evaluate_k_candidates, get_feature_selector
from src.modeling import build_pipeline, evaluate_cv
from src.evaluation import compute_regression_metrics, diagnose_residuals
from src.intervals import SplitConformalPredictor

print("All HW2 modules imported successfully!")

## Phase 2: Data Understanding (資料探索與 EDA)
載入主題 A (Kaggle s4e5 洪水機率預測資料集)，檢查型別、缺失值並排除純識別碼 `id`。

In [ ]:
df, target_col, feature_cols, num_cols, cat_cols = load_dataset('flood')
print(f"Total rows: {len(df):,}, Features count: {len(feature_cols)}")
print("Target:", target_col)
print("Sample features:", feature_cols[:5])
df.head()

## Phase 3: Data Preparation (資料準備與嚴防洩漏)
使用 `split_data` 執行 80/20 Hold-out 切分，建立 `ColumnTransformer` 串接 `StandardScaler`。

In [ ]:
X_train, X_test, y_train, y_test = split_data(df, target_col, feature_cols, test_size=0.20, random_state=42)
print(f"Train size: {len(X_train):,}, Test size: {len(X_test):,}")

preprocessor = build_preprocessor(num_cols, cat_cols)
X_train_trans = preprocessor.fit_transform(X_train)
X_test_trans = preprocessor.transform(X_test)
print("Transformed X_train shape:", X_train_trans.shape)
print("Transformed X_test shape:", X_test_trans.shape)

## Phase 4: Modeling & Feature Selection (特徵選擇與建模)
以 5 折交叉驗證評估候選特徵數 $k$，並擬合全特徵多元線性回歸、選後回歸 ($k=15$)、Ridge 與 Lasso。

In [ ]:
# Load precomputed 5-fold CV comparison table
cv_comp_df = pd.read_csv(BASE_DIR / 'reports' / 'gate3_model_comparison.csv')
cv_comp_df

## Phase 5: Evaluation & 95% Prediction Intervals (模型評估與預測區間)
在完全隔離的測試集 (223,592 筆) 計算 MAE、MSE、RMSE、R² 與 Adjusted R²，並檢驗 Split Conformal 95% 預測區間覆蓋率。

In [ ]:
test_metrics_df = pd.read_csv(BASE_DIR / 'reports' / 'gate4_test_metrics.csv')
test_metrics_df

### 視覺化展示：實際值 vs 預測值與 95% 預測區間 (PI) / 信賴區間 (CI)

In [ ]:
from IPython.display import Image, display
display(Image(filename=str(BASE_DIR / 'figures' / 'g4_actual_vs_predicted_intervals.png')))

## Phase 6: Deployment (部署與互動介面)
啟動 Streamlit 網頁應用程式：
```bash
streamlit run app.py
```